In [1]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Use SQLite backend instead of filesystem
db_path = os.path.abspath("../mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("PatrolIQ_Clustering")

df = pd.read_csv('../data/crime_final.csv')
geo = df[['Latitude', 'Longitude']].values
scaler = StandardScaler()
geo_scaled = scaler.fit_transform(geo)
print("Ready")

Ready


In [2]:
for k in [5, 6, 7, 8, 9]:
    with mlflow.start_run(run_name=f"KMeans_K{k}"):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(geo_scaled)
        
        sil = silhouette_score(geo_scaled, labels, sample_size=10000)
        db  = davies_bouldin_score(geo_scaled, labels)
        
        mlflow.log_params({"algorithm": "KMeans", "n_clusters": k})
        mlflow.log_metrics({"silhouette_score": sil, "davies_bouldin": db})
        mlflow.sklearn.log_model(km, "model")
        
        print(f"K={k}  Silhouette={sil:.4f}  DB={db:.4f}")

2026/07/02 13:34:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


K=5  Silhouette=0.4036  DB=0.8311


2026/07/02 13:34:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


K=6  Silhouette=0.3993  DB=0.8323


2026/07/02 13:34:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


K=7  Silhouette=0.3977  DB=0.8451


2026/07/02 13:34:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


K=8  Silhouette=0.3971  DB=0.8043


2026/07/02 13:34:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


K=9  Silhouette=0.4028  DB=0.7977


In [3]:
for eps in [0.05, 0.08, 0.1]:
    with mlflow.start_run(run_name=f"DBSCAN_eps{eps}"):
        db_model = DBSCAN(eps=eps, min_samples=50, n_jobs=-1)
        labels = db_model.fit_predict(geo_scaled)
        
        mask = labels != -1
        n_clusters = len(set(labels)) - 1
        noise_pct = (~mask).mean() * 100
        
        mlflow.log_params({"algorithm": "DBSCAN", "eps": eps, "min_samples": 50})
        mlflow.log_metric("n_clusters", n_clusters)
        mlflow.log_metric("noise_pct", noise_pct)
        
        if mask.sum() > 1 and len(set(labels[mask])) > 1:
            sil = silhouette_score(geo_scaled[mask], labels[mask], sample_size=10000)
            mlflow.log_metric("silhouette_score", sil)
            print(f"eps={eps}  Clusters={n_clusters}  Noise={noise_pct:.1f}%  Silhouette={sil:.4f}")
        else:
            print(f"eps={eps}  Clusters={n_clusters}  Noise={noise_pct:.1f}%  (not enough clusters to score)")

eps=0.05  Clusters=17  Noise=0.2%  Silhouette=-0.1674
eps=0.08  Clusters=8  Noise=0.0%  Silhouette=0.0735
eps=0.1  Clusters=3  Noise=0.0%  Silhouette=0.5508


In [4]:
sample_idx = np.random.choice(len(geo_scaled), size=10000, replace=False)
geo_sample = geo_scaled[sample_idx]

for k in [5, 7, 9]:
    with mlflow.start_run(run_name=f"Hierarchical_K{k}"):
        hc = AgglomerativeClustering(n_clusters=k)
        labels = hc.fit_predict(geo_sample)
        
        sil = silhouette_score(geo_sample, labels)
        db  = davies_bouldin_score(geo_sample, labels)
        
        mlflow.log_params({"algorithm": "Hierarchical", "n_clusters": k})
        mlflow.log_metrics({"silhouette_score": sil, "davies_bouldin": db})
        
        print(f"K={k}  Silhouette={sil:.4f}  DB={db:.4f}")

K=5  Silhouette=0.3874  DB=0.7522
K=7  Silhouette=0.3479  DB=0.9329
K=9  Silhouette=0.3460  DB=0.8444


In [9]:
print("Run this in your terminal (with venv active):")
print("mlflow ui --backend-store-uri ../mlruns")
print("\nThen open: http://localhost:5001")

Run this in your terminal (with venv active):
mlflow ui --backend-store-uri ../mlruns

Then open: http://localhost:5001


In [10]:
import subprocess
import webbrowser
import time
import os

db_path = os.path.abspath("../mlflow.db")

# Launch MLflow UI in background
subprocess.Popen(
    f'mlflow ui --backend-store-uri sqlite:///{db_path} --port 5001',
    shell=True
)

time.sleep(3)  # wait for server to start
webbrowser.open("http://127.0.0.1:5001")
print("MLflow UI launched → http://127.0.0.1:5001")

MLflow UI launched → http://127.0.0.1:5001
